<a href="https://colab.research.google.com/github/HamzaYaqoob1025/Delete-this-/blob/master/torrent-to-google-drive-downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader

**Important Note:** To get more disk space:
> Go to Runtime -> Change Runtime and give GPU as the Hardware Accelerator.  You will get around 384GB to download any torrent you want.

### Install libtorrent and Initialize Session

In [11]:
!apt update
!apt install python3-libtorrent

import libtorrent as lt

ses = lt.session()
ses.listen_on(6881, 6891)
downloads = []

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,074 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
40 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of

ImportError: /usr/local/lib/python3.12/dist-packages/libtorrent.so: undefined symbol: PyUnicode_AsUnicode

# Importing a library that is not in Colaboratory

To import a library that's not in Colaboratory by default, you can use `!pip install` or `!apt-get install`.

### Mount Google Drive
To stream files we need to mount Google Drive.

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


### Add From Torrent File
You can run this cell to add more files as many times as you want

In [4]:
from google.colab import files

source = files.upload()
params = {
    "save_path": "/content/drive/My Drive/Torrent",
    "ti": lt.torrent_info(list(source.keys())[0]),
}
downloads.append(ses.add_torrent(params))

Saving Shingeki no Bahamut Virgin Soul.torrent to Shingeki no Bahamut Virgin Soul.torrent


NameError: name 'lt' is not defined

### Add From Magnet Link
You can run this cell to add more files as many times as you want

In [ ]:
params = {"save_path": "/content/drive/My Drive/Torrent"}

while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break
    downloads.append(
        lt.add_magnet_uri(ses, magnet_link, params)
    )

### Start Download
Source: https://stackoverflow.com/a/5494823/7957705 and [#3 issue](https://github.com/FKLC/Torrent-To-Google-Drive-Downloader/issues/3) which refers to this [stackoverflow question](https://stackoverflow.com/a/6053350/7957705)

In [6]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "queued",
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}
download_bars = [
    widgets.FloatSlider(
        step=0.01, disabled=True, layout=layout, style=style
    )
    for _ in downloads
]
display(*download_bars)

while downloads:
    next_shift = 0
    for index, download in enumerate(downloads[:]):
        bar = download_bars[index + next_shift]
        if not download.is_seed():
            s = download.status()

            bar.description = " ".join(
                [
                    download.name(),
                    str(s.download_rate / 1000),
                    "kB/s",
                    state_str[s.state],
                ]
            )
            bar.value = s.progress * 100
        else:
            next_shift -= 1
            ses.remove_torrent(download)
            downloads.remove(download)
            bar.close() # Seems to be not working in Colab (see https://github.com/googlecolab/colabtools/issues/726#issue-486731758)
            download_bars.remove(bar)
            print(download.name(), "complete")
    time.sleep(1)

NameError: name 'downloads' is not defined

# Task
Install `libtorrent` in the current environment.

## Install build dependencies

### Subtask:
Install necessary tools and libraries for compiling `libtorrent` from source.


**Reasoning**:
Install the necessary build tools and libraries for compiling libtorrent from source as requested in the instructions.



In [12]:
!apt update
!apt install build-essential pkg-config libboost-python-dev libboost-system-dev python3-dev

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,120 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
40 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of

## Download and extract libtorrent source code

### Subtask:
Download the `libtorrent` source code and extract it.


**Reasoning**:
Download the libtorrent source code tarball using wget and extract it using tar.



In [13]:
!wget https://github.com/arvidn/libtorrent/releases/download/v2.0.9/libtorrent-rasterbar-2.0.9.tar.gz
!tar -xzf libtorrent-rasterbar-2.0.9.tar.gz

--2025-10-18 22:18:37--  https://github.com/arvidn/libtorrent/releases/download/v2.0.9/libtorrent-rasterbar-2.0.9.tar.gz
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/36781769/160ebaf9-1c4e-45cb-a39b-509700cb855a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-10-18T23%3A13%3A35Z&rscd=attachment%3B+filename%3Dlibtorrent-rasterbar-2.0.9.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-10-18T22%3A13%3A35Z&ske=2025-10-18T23%3A13%3A35Z&sks=b&skv=2018-11-09&sig=RlJZVtRgGvpkMb3P5VzpQNfgwUKnvBLlFcNRI6SCxX0%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2MDgyNjIxNywibmJmIjoxNzYwODI1OTE3LCJwYXRoIjoic

## Configure and compile libtorrent

### Subtask:
Configure the build process with appropriate flags for Python 3 compatibility and compile the library.


**Reasoning**:
Change directory to the extracted source and configure the build with Python 3 bindings and then compile the source code.



In [ ]:
%cd libtorrent-rasterbar-2.0.9
!./configure --enable-python-binding PYTHON="python3"
!make -j$(nproc)

/content/libtorrent-rasterbar-2.0.9
/bin/bash: line 1: ./configure: No such file or directory
BOOST_ROOT="" b2 release link=shared crypto=openssl warnings=off address-model=64
CXXFLAGS =
LDFLAGS =
OS = LINUX
...patience...
...found 1217 targets...
...updating 189 targets...
gcc.compile.c++ bin/gcc-11/release/address-model-64/cxxstd-14-iso/threading-multi/visibility-hidden/src/alert_manager.o
gcc.compile.c++ bin/gcc-11/release/address-model-64/cxxstd-14-iso/threading-multi/visibility-hidden/src/announce_entry.o
gcc.compile.c++ bin/gcc-11/release/address-model-64/cxxstd-14-iso/threading-multi/visibility-hidden/src/assert.o
gcc.compile.c++ bin/gcc-11/release/address-model-64/cxxstd-14-iso/threading-multi/visibility-hidden/src/bandwidth_limit.o
gcc.compile.c++ bin/gcc-11/release/address-model-64/cxxstd-14-iso/threading-multi/visibility-hidden/src/bandwidth_manager.o
gcc.compile.c++ bin/gcc-11/release/address-model-64/cxxstd-14-iso/threading-multi/visibility-hidden/src/bandwidth_queue_entry